# Sesión 3.03 · Weaviate local mediante SDK nativo

Weaviate será una alternativa local de esta sesión. Lo ejecutaremos en Docker como un servicio independiente y el notebook actuará como cliente mediante su SDK nativo. Esa separación permite observar una arquitectura real: la aplicación no es la base de datos, sino que se comunica con ella mediante una API.

El laboratorio mantendrá exactamente el contrato común: utilizará los embeddings ya calculados, no delegará la inferencia en el proveedor y no añadirá LangChain. Si el ranking difiere del oráculo exacto, podremos investigar el esquema, el índice, los filtros, la visibilidad de las escrituras o la semántica de la respuesta sin mezclar esas causas con otro encoder o framework.


<a id="s03-weaviate-indice"></a>

## Índice de contenidos

1. [Configuración del entorno y del índice](#s03-weaviate-configuracion)
2. [Ingesta](#s03-weaviate-ingesta)
3. [Verificación de la carga](#s03-weaviate-verificacion)
4. [Resultados de búsqueda](#s03-weaviate-resultados)
5. [Búsqueda sin filtro](#s03-weaviate-busqueda-sin-filtro)
6. [Búsqueda filtrada](#s03-weaviate-busqueda-filtrada)
7. [Operaciones CRUD](#s03-weaviate-operaciones-crud)
8. [Informe de ejecución](#s03-weaviate-informe)
9. [Consideraciones específicas](#s03-weaviate-consideraciones)
10. [Próximos pasos](#s03-weaviate-proximos-pasos)
11. [Limpieza](#s03-weaviate-limpieza)

## Objetivo del laboratorio

También registraremos tiempos, pero solo para describir esta ejecución concreta. No los utilizaremos para comparar proveedores, porque una llamada a un servicio gestionado y una ejecución local no comparten red, hardware, caché ni condiciones de carga.

El recorrido comenzará con un **preflight**. Si faltan credenciales, el índice no existe o su configuración no coincide con el contrato esperado, el notebook se detendrá y mostrará la causa. Continuar después de un fallo con una variable vacía como `results = []` ocultaría información importante: no podríamos distinguir entre una búsqueda que realmente no encontró vecinos y una consulta que nunca llegó a ejecutarse.

In [ ]:
from pathlib import Path
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vector_database_session import (
    ProviderRun,
    SearchHit,
    evaluate_run,
    exact_top_k,
    iter_record_batches,
    load_session_data,
    record_id_for_product,
    validate_resource_name,
    wait_until,
    write_provider_run,
)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env")
data = load_session_data(memory_map=True)
TELEVISOR_QUERY_ID = "semantic-101352"
TALADRO_QUERY_ID = "semantic-100455"
TOP_K = 10

In [ ]:
RESOURCE_NAME = validate_resource_name(os.getenv("WEAVIATE_COLLECTION", "BbddVectorialesS03Products"))
BATCH_SIZE = 500
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "esci-es-s03")
TEMPORARY_TEST_ID = record_id_for_product("S03-TEMPORARY-TEST")
assert os.getenv("S03_ALLOW_REMOTE_CLEANUP", "false").lower() != "true", "La limpieza remota no pertenece a la ejecución docente"
print({"python": platform.python_version(), "resource": RESOURCE_NAME, "batch_size": BATCH_SIZE})

<a id="s03-weaviate-configuracion"></a>

## 1. Configuración del entorno y del índice

Antes de crear nada abrimos una conexión con el servidor local y comprobamos que está listo. El cliente v4 mantiene una conexión gRPC, por lo que deberá cerrarse al terminar. Importar el SDK no demuestra que el servicio esté disponible: esa comprobación pertenece al preflight.

Weaviate permite elegir por colección entre HNSW, Flat, Dynamic y HFresh. Para este experimento elegimos HNSW de manera explícita: coseno, max_connections=24, ef_construction=120 y ef=128. Así podemos relacionar el laboratorio con los compromisos entre recall, coste de construcción y latencia vistos en la sesión 2.

Que la consulta filtrada devuelva resultados correctos no revela por sí solo el camino físico seguido. Weaviate puede adaptar su estrategia cuando el conjunto permitido es restrictivo. Por eso compararemos el ranking con el oráculo exacto condicionado, no solo la marca de los resultados.


In [ ]:
#NOTE: Ejecuta esta celda solo si no has levantado el contenedor de Weaviate antes.
!docker compose -f ../deploy/weaviate/compose.yaml up -d

In [ ]:
import weaviate
import weaviate.classes as wvc
from weaviate import __version__ as provider_version

host = os.getenv("WEAVIATE_HOST", "localhost")
http_port = int(os.getenv("WEAVIATE_HTTP_PORT", "8080"))
grpc_port = int(os.getenv("WEAVIATE_GRPC_PORT", "50051"))

client = weaviate.connect_to_local(host=host, port=http_port, grpc_port=grpc_port)

if not client.is_ready():
    client.close()
    raise RuntimeError("Weaviate responde, pero todavía no está ready")
if not client.collections.exists(RESOURCE_NAME):
    client.collections.create(
        name=RESOURCE_NAME,
        properties=[
            wvc.config.Property(name="record_id", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="product_id", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="vector_id", data_type=wvc.config.DataType.INT),
            wvc.config.Property(name="title", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="brand", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="color", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="locale", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        ],
        vector_config=wvc.config.Configure.Vectors.self_provided(
            vector_index_config=wvc.config.Configure.VectorIndex.hnsw(
                distance_metric=wvc.config.VectorDistances.COSINE,
                max_connections=24,  # M
                ef_construction=120,
                ef=128,
            )
        ),
    )
    
collection = client.collections.use(RESOURCE_NAME)
print({"version": provider_version, "ready": client.is_ready(), "target": f"{host}:{http_port}/{grpc_port}"})

Antes de ingerir los vectores, conviene inspeccionar la colección y su configuración de vectorización. En este laboratorio aportamos el vector desde el cliente; activar por error otro vectorizador produciría una segunda representación y cambiaría el experimento sin necesidad de que aparezca un error.

También interesa distinguir entre el esquema de propiedades, el índice vectorial y las estructuras que ayudan a filtrar. Cada decisión responde a una pregunta diferente: qué datos se almacenan, cómo se recuperan vecinos y cómo se restringe el universo de candidatos.


<a id="s03-weaviate-ingesta"></a>

## 2. Ingesta

Cada producto utilizará el mismo UUIDv5 en los cinco motores. Como el identificador se genera de forma determinista a partir de `product_id`, volver a ejecutar la ingesta produce exactamente los mismos IDs.

La operación `upsert` aprovecha esa propiedad: si el registro no existe, lo inserta; si ya estaba presente, lo reemplaza o actualiza según la semántica del motor. Gracias a ello, el notebook puede reanudarse después de una interrupción sin crear duplicados.

Esto no significa que toda la tubería ofrezca una garantía de *exactly once*. Si el proceso falla a mitad de la carga, algunos batches pueden haber sido confirmados y otros no. La idempotencia no evita ese estado parcial, pero hace que repetir la operación sea seguro: los lotes ya escritos se procesarán de nuevo sobre los mismos IDs y los pendientes podrán completarse.

Utilizaremos batches de 500 registros como punto de partida para la práctica. No debe interpretarse como un tamaño óptimo universal. En producción, el valor adecuado depende de los límites de cada petición, la memoria disponible en el cliente, el tamaño de los metadatos, la latencia de red y el grado de concurrencia.

Registraremos el tiempo total de ingesta para describir esta ejecución y detectar posibles anomalías. No lo utilizaremos para comparar Pinecone con motores ejecutados en otros entornos, porque las condiciones de hardware, red y despliegue no son equivalentes.

In [ ]:
ingestion_started = time.perf_counter()

for batch_number, records in enumerate(iter_record_batches(data, batch_size=BATCH_SIZE), start=1):
    with collection.batch.fixed_size(batch_size=BATCH_SIZE) as batch:
        for record in records:
            batch.add_object(
                uuid=record.record_id,
                properties={**record.flat_metadata(), "text": record.text},
                vector=record.embedding,
            )
    if collection.batch.failed_objects:
        raise RuntimeError(f"Weaviate rejected {len(collection.batch.failed_objects)} objects")
    if batch_number % 20 == 0:
        print(f"{batch_number * BATCH_SIZE:,} objetos confirmados")
        
ingestion_ms = (time.perf_counter() - ingestion_started) * 1000

<a id="s03-weaviate-verificacion"></a>

## 3. Verificación de la carga

Que el servidor haya aceptado 50.000 escrituras no significa todavía que los 50.000 registros estén disponibles para búsqueda. La ingesta y la visibilidad forman parte de momentos distintos del ciclo de escritura.

Por eso no daremos por completada la carga a partir del número de batches enviados ni de las respuestas correctas del cliente. Consultaremos el estado que expone el propio motor y comprobaremos cuántos registros reconoce dentro del namespace.

En un servicio con visibilidad eventual, ese recuento puede tardar unos instantes en alcanzar el valor esperado. Esperaremos de forma acotada y registraremos la evolución observada. Si se agota el plazo sin llegar a 50.000 registros visibles, el notebook fallará mostrando el último estado recibido.

De este modo distinguiremos entre tres situaciones diferentes: una escritura rechazada, una escritura aceptada pero todavía no visible y una carga completamente disponible para consulta.

In [ ]:
aggregate = collection.aggregate.over_all(total_count=True)
record_count = int(aggregate.total_count)
visibility_seconds, visibility_attempts = 0.0, 1

assert record_count == 50_000, f'Recuento inesperado: {record_count}'
print({'record_count': record_count, 'ingestion_ms': ingestion_ms, 'visible_s': visibility_seconds})

Un recuento correcto no prueba alineación de IDs, contenido del payload ni calidad del ranking. Sí descarta una clase importante de explicaciones: batches perdidos o namespace equivocado. La validación completa combina invariantes, búsquedas conocidas y mutaciones.

<a id="s03-weaviate-resultados"></a>

## 4. Resultados de búsqueda

Cada motor devuelve sus resultados con una estructura propia. Para poder comparar rankings y mostrar una tabla común, traduciremos esas respuestas a un modelo compartido llamado `SearchHit`.

Ese modelo conservará el ID recuperado, pero también tres campos necesarios para interpretar correctamente la puntuación: `native_score`, `score_kind` y `higher_is_better`. Así sabremos si el proveedor devuelve una similitud o una distancia y en qué sentido debe ordenarse.

La normalización no convierte esas puntuaciones en magnitudes equivalentes. Una distancia de 0,2 y una similitud de 0,8 pueden inducir el mismo ranking, pero no representan la misma cantidad ni deben compararse directamente entre motores.

El objetivo es más sencillo: evitar repetir cinco veces el código de evaluación y trabajar con una interfaz común sin ocultar la semántica original de cada proveedor.

In [ ]:
def native_search(query_vector, *, k=10, brand=None):
    result_filter = wvc.query.Filter.by_property("brand").equal(brand) if brand else None
    response = collection.query.near_vector(
        near_vector=np.asarray(query_vector, dtype=np.float32).tolist(),
        limit=k,
        filters=result_filter,
        return_metadata=wvc.query.MetadataQuery(distance=True),
        return_properties=["product_id", "vector_id", "title", "brand"],
    )
    return [
        SearchHit(
            record_id=str(item.uuid),
            product_id=item.properties["product_id"],
            vector_id=int(item.properties["vector_id"]),
            title=item.properties["title"],
            brand=item.properties.get("brand", ""),
            native_score=float(item.metadata.distance),
            score_kind="distance",
            higher_is_better=False,
            rank=rank,
        )
        for rank, item in enumerate(response.objects, start=1)
    ]

<a id="s03-weaviate-busqueda-sin-filtro"></a>

## 5. Búsqueda sin filtro

Volveremos ahora a la consulta problemática del televisor. Antes de inspeccionar la tabla, conviene formular una predicción: si Weaviate reproduce fielmente el espacio generado por E5, debería devolver en primera posición el mismo producto que el oráculo exacto.

En este caso, ese producto es un mantel. El resultado es claramente poco útil para la intención de búsqueda, pero su presencia no demuestra un fallo de la base de datos. Al contrario: si el motor devuelve el mismo UUID y mantiene un `recall@10` alto frente al oráculo, estará reproduciendo correctamente una geometría semántica que ya contenía ese error.

Que cinco bases de datos distintas recuperen el mismo mantel no lo convierte en relevante. Lo convierte en una evidencia reproducible de que el problema se encuentra antes, en el encoder, en la representación del texto o en los datos con los que se construyó el espacio.

Esta distinción será central durante la inspección. La fidelidad mide cuánto respeta el motor el ranking definido por los vectores; la relevancia mide si ese ranking responde realmente a la necesidad del usuario. Ambas propiedades pueden coincidir, pero no son equivalentes.

In [ ]:
television_row = data.query_row(TELEVISOR_QUERY_ID)
television_vector = data.query_vector(TELEVISOR_QUERY_ID)

exact_hits = exact_top_k(data, television_vector, k=TOP_K)

query_started = time.perf_counter()
native_hits = native_search(television_vector, k=TOP_K)
query_ms = (time.perf_counter() - query_started) * 1000

television_evaluation = evaluate_run(exact_hits, native_hits, k=TOP_K)

display(Markdown(f"**Consulta:** {television_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in native_hits]))
television_evaluation

Interpreta `recall@10` como una medida de fidelidad algorítmica frente al oráculo exacto, no como una medida de relevancia humana. Un valor inferior a uno indica que el motor no ha reproducido por completo el top-10 de fuerza bruta, pero no identifica por sí solo la causa. La pérdida puede proceder del índice ANN, de una configuración de búsqueda demasiado restrictiva o de que algunos registros todavía no sean visibles para la consulta.

Un `recall@10` igual a uno cuenta una historia distinta. Si el ranking coincide con el oráculo y el mantel sigue apareciendo en primera posición, la base de datos ha reproducido correctamente el espacio de E5. En ese caso, el error debe atribuirse a la representación o al modelo, no al mecanismo de recuperación.

> **Pregunta.** Para distinguir una pérdida ANN estable de una escritura todavía no visible, repetiría la misma consulta después de verificar por ID y por recuento que todos los registros esperados están disponibles. Si el recall mejora a medida que se completa la visibilidad, el problema era de indexación o consistencia. Si permanece estable una vez confirmado el snapshot completo, la pérdida apunta al ANN o a su configuración.

<a id="s03-weaviate-busqueda-filtrada"></a>

## 6. Búsqueda filtrada

Ejecutaremos la misma consulta sobre taladros de dos formas. La primera buscará los vecinos más próximos dentro de todo el catálogo. La segunda añadirá la condición `brand == "Einhell"` directamente a la petición enviada al motor.

Esto significa que el top-$k$ filtrado debe calcularse sobre el conjunto de productos Einhell, no sobre los diez primeros resultados de la búsqueda global. No recuperaremos primero diez vecinos y eliminaremos después los que pertenezcan a otras marcas, porque ese procedimiento podría devolver menos resultados y perder candidatos válidos situados más abajo en el ranking general.

La comparación permitirá observar cómo cambia el universo de búsqueda cuando el filtro forma parte del contrato del motor. También compararemos los IDs devueltos con un oráculo exacto construido sobre el mismo subconjunto de productos Einhell, de modo que podamos medir la fidelidad del ranking condicionado y no solo comprobar que todos los resultados pertenecen a la marca solicitada.

In [ ]:
drill_row = data.query_row(TALADRO_QUERY_ID)
drill_vector = data.query_vector(TALADRO_QUERY_ID)

global_drill_hits = native_search(drill_vector, k=TOP_K)
filtered_drill_hits = native_search(drill_vector, k=TOP_K, brand="Einhell")

exact_filtered_hits = exact_top_k(data, drill_vector, k=TOP_K, brand="Einhell")

filtered_evaluation = evaluate_run(exact_filtered_hits, filtered_drill_hits, k=TOP_K)

assert filtered_drill_hits and all(hit.brand == "Einhell" for hit in filtered_drill_hits)
display(Markdown(f"**Consulta:** {drill_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in global_drill_hits]))
display(pd.DataFrame([hit.as_dict() for hit in filtered_drill_hits]))
filtered_evaluation

El filtro no es una operación decorativa aplicada al final de la consulta. Cambia el conjunto sobre el que debe calcularse el ranking: buscamos los vecinos más próximos entre los productos Einhell, no los productos Einhell que hayan sobrevivido por casualidad al top-10 global.

Por eso la comprobación no termina al ver que todos los resultados pertenecen a la marca correcta. También compararemos sus IDs con el oráculo exacto construido sobre ese mismo subconjunto. Solo así podremos distinguir un filtro funcional de una búsqueda condicionada que ha perdido vecinos durante la recuperación.


<a id="s03-weaviate-operaciones-crud"></a>

## 7. Operaciones CRUD

El canary test utilizará un UUID conocido y reservado para esta sesión. Sobre ese único registro recorreremos el ciclo completo: lo insertaremos, comprobaremos que puede recuperarse por ID y mediante búsqueda, modificaremos uno de sus campos y, finalmente, lo eliminaremos.

El objetivo no es solo confirmar que el motor admite operaciones CRUD. También queremos observar cuánto tarda cada cambio en hacerse visible desde las distintas rutas de lectura. Por eso registraremos los intentos y el tiempo transcurrido hasta detectar la inserción, la actualización y el borrado.

Al terminar eliminaremos únicamente el registro temporal de prueba. No borraremos el namespace ni el índice completo, porque la prueba debe ser segura y no afectar al resto de los datos ingeridos.

Un resultado inmediato tampoco demuestra que el sistema sea fuertemente consistente en todos los casos. Solo describe lo ocurrido para esta operación, en esta ruta de lectura y durante esta ejecución concreta. La prueba aporta evidencia observable, pero no permite generalizar una garantía más amplia que la documentada por el proveedor.

In [ ]:
started = time.perf_counter()
collection.data.insert(
    uuid=TEMPORARY_TEST_ID,
    properties={
        "product_id": "S03-TEMPORARY-TEST",
        "vector_id": -1,
        "title": "Registro temporal de prueba",
        "brand": "S03",
        "color": "amarillo",
        "locale": "es",
        "text": "registro temporal"
    },
    vector=television_vector.tolist()
)

fetched = collection.query.fetch_object_by_id(TEMPORARY_TEST_ID)
assert fetched is not None
upsert_visible_s, upsert_attempts = time.perf_counter() - started, 1

collection.data.update(uuid=TEMPORARY_TEST_ID, properties={"brand": "S03-updated"})
updated = collection.query.fetch_object_by_id(TEMPORARY_TEST_ID)
assert updated.properties["brand"] == "S03-updated"
update_visible_s, update_attempts = 0.0, 1

collection.data.delete_by_id(TEMPORARY_TEST_ID)
assert collection.query.fetch_object_by_id(TEMPORARY_TEST_ID) is None
delete_visible_s, delete_attempts = 0.0, 1

mutation = {"upsert": True, "fetch_or_query": True, "update": True, "delete": True}
temporary_test_visibility = {
    "upsert_seconds": upsert_visible_s,
    "upsert_attempts": upsert_attempts,
    "update_seconds": update_visible_s,
    "update_attempts": update_attempts,
    "delete_seconds": delete_visible_s,
    "delete_attempts": delete_attempts,
}
temporary_test_visibility

<a id="s03-weaviate-informe"></a>

## 8. Informe de ejecución

El informe final conservará la información necesaria para reconstruir e interpretar esta ejecución: versiones del cliente y del servicio, destino consultado, número de registros visibles, tiempos observados, scores nativos, IDs recuperados y resultado completo del canary test.

Los tiempos deben leerse como una descripción del experimento, no como un benchmark entre proveedores. No hemos controlado calentamiento, concurrencia, red, hardware ni carga de fondo, y los motores ni siquiera se ejecutan en infraestructuras comparables. Una diferencia de milisegundos no permite concluir qué solución sería más rápida en producción.

Sí podemos comparar aquello que mantuvimos constante: el contrato funcional, los filtros aplicados, la semántica de las respuestas y el `recall@10` frente al mismo oráculo exacto. Esa evidencia permite saber si cada motor respeta los datos, las condiciones y el espacio vectorial definidos para la práctica.

Persistir estos resultados no convierte una única ejecución en una verdad general. Su valor está en dejar una traza reproducible: qué se probó, bajo qué configuración y qué ocurrió exactamente.

In [ ]:
run = ProviderRun(
    provider="weaviate",
    provider_version=str(provider_version),
    target="Docker HTTP/gRPC localhost",
    resource=RESOURCE_NAME,
    record_count=record_count,
    score_kind="distance",
    higher_is_better=False,
    query_id=TELEVISOR_QUERY_ID,
    query_text=str(television_row["query_text"]),
    top_k=TOP_K,
    hits=native_hits,
    exact_record_ids=television_evaluation["exact_record_ids"],
    recall_at_k=float(television_evaluation["recall_at_k"]),
    filtered_hits=filtered_drill_hits,
    filtered_recall_at_k=float(filtered_evaluation["recall_at_k"]),
    durations_ms={"ingestion": ingestion_ms, "television_query": query_ms},
    mutation=mutation,
    visibility={"initial_count_seconds": visibility_seconds, "initial_count_attempts": visibility_attempts, **temporary_test_visibility},
    notes=["Los tiempos describen esta ejecución y no forman un ranking entre proveedores."],
)
report_path = write_provider_run(run)
print(f"Informe escrito en {report_path.relative_to(PROJECT_ROOT)}")

<a id="s03-weaviate-consideraciones"></a>

## 9. Consideraciones específicas de Weaviate

Weaviate trabaja con objetos tipados y permite elegir explícitamente el tipo de índice vectorial. HNSW es la elección de este laboratorio, pero no es la única: Flat sirve como referencia exacta para conjuntos pequeños, Dynamic cambia de Flat a HNSW al crecer y HFresh prioriza el uso de memoria.

Antes de elegir Weaviate para producción habría que probar la configuración elegida con el volumen y los filtros reales, además de documentar backups, autenticación, aislamiento, upgrades, observabilidad, consistencia y recuperación ante fallos.


<a id="s03-weaviate-proximos-pasos"></a>

## 10. Próximos pasos

Cada motor ofrece capacidades que van más allá de la búsqueda densa utilizada en esta comparación: recuperación híbrida, vectores sparse, múltiples representaciones por documento, cuantización, reranking, inferencia integrada o mecanismos específicos de multitenancy.

No las incorporaremos todavía porque cambiarían la pregunta del experimento. Si un proveedor utilizara búsqueda híbrida y otro únicamente embeddings densos, una diferencia en los resultados ya no podría atribuirse con claridad al motor, al índice o a la estrategia de recuperación.

La siguiente ampliación debería comenzar siempre por un requisito medible. Por ejemplo, mejorar el recall de consultas con referencias exactas, reducir memoria o aislar tenants con una garantía concreta. A partir de ahí se añadirá una sola capacidad cada vez y se repetirá la evaluación, de modo que podamos observar qué mejora introduce y qué coste añade.

El notebook de LangChain reutilizará las colecciones ya creadas en Chroma y Qdrant. No repetirá la ingesta ni construirá una copia paralela de los datos. Si una capa de abstracción necesita duplicar toda la colección para poder conectarse, ya no estaría envolviendo el mismo sistema: estaría creando otro despliegue y alterando la comparación.

<a id="s03-weaviate-limpieza"></a>

## 11. Limpieza

Hasta este punto hemos creado una colección de Weaviate y hemos cargado en ella el catálogo de productos. La celda siguiente permite eliminar esa colección cuando quieras terminar la práctica o empezar otra vez desde cero. Al ejecutarla se borrarán los vectores, los metadatos y el índice asociados a esa colección.

Para evitar un borrado accidental, la operación está desactivada inicialmente. Si quieres activarla, escribe en el archivo .env una confirmación con este formato:

    S03_CONFIRM_CLEANUP=DELETE:<nombre-de-la-colección>

Después guarda el archivo, vuelve a ejecutar la celda inicial de configuración del notebook y ejecuta la celda de limpieza. Si el valor no coincide exactamente con el nombre de la colección cargada, el notebook no borrará nada.

Esta acción elimina los datos de la colección, pero mantiene el contenedor y el volumen de Docker. Es lo normal si quieres volver a ejecutar el laboratorio más adelante. Si también quieres eliminar el entorno local de Weaviate y sus datos persistidos, ejecuta en una terminal:

    docker compose -f deploy/weaviate/compose.yaml down --volumes

El último comando también borra el volumen de Docker. Úsalo solo si realmente quieres reiniciar el motor desde cero.


In [ ]:
confirmation = os.getenv("S03_CONFIRM_CLEANUP", "")
expected_confirmation = f"DELETE:{RESOURCE_NAME}"

if confirmation != expected_confirmation:
    print({
        "cleanup": "omitida",
        "motivo": "Define S03_CONFIRM_CLEANUP con la confirmación exacta para habilitarla.",
        "confirmacion_requerida": expected_confirmation,
    })
else:
    client.collections.delete(RESOURCE_NAME)
    client.close()
    print({"cleanup": "completada", "recurso_eliminado": RESOURCE_NAME})


**Ejecuta también la siguiente celda si quieres eliminar el contenedor de Docker, así como la imagen de Weaviate y el volumen persistente:**

In [ ]:
!docker compose -f ../deploy/weaviate/compose.yaml down --volumes --remove-orphans --rmi all